In this chaper we will work with unstructured text data, understading ways in which we can engineer columnar features out of a text corpous. We will compare how differne approaches may impach how much context being extracted from a text and how to balance the need for context, without too many features being created.


### Encoding Text Data:
Text data is complicated as its in the form of text and machine learning algorithms cannot learn text and they need only numbers. Hence we need to encoded or tokenize these value for our machine learning algorithms to understand. 

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

speech_df = pd.read_csv('data/inaug_speeches.csv', encoding='latin1')
speech_df.head()


,Unnamed: 0,Name,Inaugural Address,Date,text
0,4,George Washington,First Inaugural Address,"Thursday, April 30, 1789",Fellow-Citizens of the Senate and o...
1,5,George Washington,Second Inaugural Address,"Monday, March 4, 1793",Fellow Citizens: I AM again cal...
2,6,John Adams,Inaugural Address,"Saturday, March 4, 1797","WHEN it was first perceived, in ..."
3,7,Thomas Jefferson,First Inaugural Address,"Wednesday, March 4, 1801",Friends and Fellow-Citizens: CA...
4,8,Thomas Jefferson,Second Inaugural Address,"Monday, March 4, 1805","PROCEEDING, fellow-citizens, to ..."


Before any text analytics be performed, we should remove some unwanted spaces and non-letter charecters.

In [10]:
# Include all Charecters
speech_df['text'] = speech_df['text'].str.replace('[^a-ZA-Z]',' ')
print(speech_df['text'][0])


           Fellow-Citizens of the Senate and of the House of Representatives:    AMONG the vicissitudes incident to life no event could have filled me with greater anxieties than that of which the notification was transmitted by your order, and received on the   th day of the present month. On the one hand, I was summoned by my country, whose voice I can never hear but with veneration and love, from a retreat which I had chosen with the fondest predilection, and, in my flattering hopes, with an immutable decision, as the asylum of my declining years<U+0097>a retreat which was rendered every day more necessary as well as more dear to me by the addition of habit to inclination, and of frequent interruptions in my health to the gradual waste committed on it by time. On the other hand, the magnitude and difficulty of the trust to which the voice of my country called me, being sufficient to awaken in the wisest and most experienced of her citizens a distrustful scrutiny into his qualificati

Next we will standadrize the data by using converting it into lower case values as shown below:

In [12]:
speech_df['char_cnt'] = speech_df['text'].str.len()
print(speech_df['char_cnt'].head())

0     8654
1      819
2    13928
3    10184
4    12961
Name: char_cnt, dtype: int64


We should also work on word count to find repeated words for analytics

In [ ]:
speech_df['word_count'] = speech_df['text'].str.split()
speech_df['word_count'].head(1)

0    [Fellow-Citizens, of, the, Senate, and, of, th...
Name: word_count, dtype: object

In [15]:
speech_df['word_splits'] = speech_df['text'].str.split().str.len()
print(speech_df['word_splits'].head(1))

0    1427
Name: word_splits, dtype: int64


In [16]:
speech_df['avg_word_len'] = speech_df['char_cnt'] /speech_df['word_count']
print(speech_df['avg_word_len'].head(1))

0    6.064471
Name: avg_word_len, dtype: float64


One method of encoding is `Text to Columns` . Here we count the number of words that are repeated and stored it in the a new column. Lets see how it done:

In [17]:
from  sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()
print(cv)

CountVectorizer()


We can specify how specific value that we need using 

`min_df`: Minimum fraction of documents the word must occur.

`max_df`: Maximum fraction of documents the word can occur.

In [18]:
cv = CountVectorizer(min_df =0.1, max_df=0.9)
print(cv)

CountVectorizer(max_df=0.9, min_df=0.1)


In [22]:
cv.fit(speech_df['text'])
cv_transformed = cv.transform(speech_df['text'])
print(cv_transformed)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 28024 stored elements and shape (58, 1961)>
  Coords	Values
  (0, 1)	1
  (0, 16)	1
  (0, 26)	1
  (0, 30)	1
  (0, 36)	1
  (0, 38)	1
  (0, 48)	2
  (0, 51)	1
  (0, 55)	1
  (0, 56)	1
  (0, 58)	1
  (0, 60)	1
  (0, 64)	1
  (0, 68)	1
  (0, 69)	1
  (0, 81)	1
  (0, 88)	1
  (0, 91)	1
  (0, 97)	1
  (0, 101)	2
  (0, 104)	1
  (0, 109)	1
  (0, 110)	1
  (0, 115)	1
  (0, 131)	1
  :	:
  (57, 1855)	1
  (57, 1872)	2
  (57, 1884)	1
  (57, 1886)	1
  (57, 1890)	1
  (57, 1894)	1
  (57, 1895)	2
  (57, 1898)	2
  (57, 1903)	4
  (57, 1907)	1
  (57, 1909)	1
  (57, 1912)	1
  (57, 1914)	3
  (57, 1918)	3
  (57, 1919)	6
  (57, 1921)	1
  (57, 1931)	1
  (57, 1940)	2
  (57, 1942)	1
  (57, 1943)	1
  (57, 1952)	2
  (57, 1953)	1
  (57, 1956)	14
  (57, 1957)	1
  (57, 1958)	11


In [24]:
feature_names = cv.get_feature_names_out()
print(feature_names)

['0092' '0097' 'abandon' ... 'your' 'zeal' 'zealously']


In [28]:
cv_df = pd.DataFrame(cv_transformed.toarray(),
                     columns=cv.get_feature_names_out()).add_prefix('Counts_')
print(cv_df.head())

   Counts_0092  Counts_0097  Counts_abandon  Counts_abiding  Counts_ability  \
0            0            1               0               0               0   
1            0            0               0               0               0   
2            0            1               0               0               0   
3            0            4               1               0               0   
4            0            1               0               0               0   

   Counts_able  Counts_about  Counts_above  Counts_abroad  Counts_absolute  \
0            0             0             0              0                0   
1            0             1             0              0                0   
2            0             0             0              1                0   
3            0             1             1              1                1   
4            1             0             0              0                0   

   ...  Counts_year  Counts_years  Counts_yes  Counts_ye

In [29]:
speech_df = pd.concat([speech_df, cv_df], axis=1, sort=False)
print(speech_df.head(10))

   Unnamed: 0               Name         Inaugural Address  \
0           4  George Washington   First Inaugural Address   
1           5  George Washington  Second Inaugural Address   
2           6         John Adams         Inaugural Address   
3           7   Thomas Jefferson   First Inaugural Address   
4           8   Thomas Jefferson  Second Inaugural Address   
5           9      James Madison   First Inaugural Address   
6          10      James Madison  Second Inaugural Address   
7          11       James Monroe   First Inaugural Address   
8          12       James Monroe  Second Inaugural Address   
9          13  John Quincy Adams         Inaugural Address   

                       Date  \
0  Thursday, April 30, 1789   
1     Monday, March 4, 1793   
2   Saturday, March 4, 1797   
3  Wednesday, March 4, 1801   
4     Monday, March 4, 1805   
5   Saturday, March 4, 1809   
6   Thursday, March 4, 1813   
7    Tuesday, March 4, 1817   
8     Monday, March 5, 1821   
9     F

### Tf-ldf Represenataion

Term Frequency - Document Frequency representation is one of most ways of normalization of text encoding:


```math
(TF-IDF) = 
(Count of Word Occurances/ Total Number words in the document)]
/
[log(Number of docs word is in / Total number of docs)]

```

In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer

tv = TfidfVectorizer()
print(tv)

TfidfVectorizer()


Similar to Text Vectorizer, We can also use some functions to make sure this extracts whats needed.

`max features` : will control the count of the words

`stopwords` : list of words that can be fed in order to be neglected or ingnored.

In [31]:
tv.fit(speech_df['text'])
train_tv_transformed = tv.transform(speech_df['text'])

train_tv_Df = pd.DataFrame(train_tv_transformed.toarray(),
                           columns=tv.get_feature_names_out())\
                           .add_prefix('TFIDF_')
train_speech_df = pd.concat([speech_df, train_tv_Df], axis=1,sort=False)

### Bag of words and N-Grams

Untill now we have selected words from the list like a bag of words , without any order or value. This is causes some issues as we will not be able view the context words. One method of the method is to retain two words(bi-gram) or retain three words(tri-grams)


In [34]:
tv_bi_gram_vec = TfidfVectorizer(ngram_range=(2,2))
tv_bi_gram = tv_bi_gram_vec\
            .fit_transform(speech_df['text'])
print(tv_bi_gram)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 108867 stored elements and shape (58, 64964)>
  Coords	Values
  (0, 19986)	0.027209882101322972
  (0, 11339)	0.015738946034079566
  (0, 38036)	0.14025555431263215
  (0, 54833)	0.020198993884725586
  (0, 47918)	0.027903317660816346
  (0, 3990)	0.020196116554575896
  (0, 53870)	0.048642022219793184
  (0, 25587)	0.048642022219793184
  (0, 37865)	0.05177173446787956
  (0, 45924)	0.030746754335423444
  (0, 2500)	0.013034380782886075
  (0, 55247)	0.027903317660816346
  (0, 60739)	0.030746754335423444
  (0, 27023)	0.024321011109896592
  (0, 58013)	0.030746754335423444
  (0, 31197)	0.030746754335423444
  (0, 35612)	0.030746754335423444
  (0, 18251)	0.030746754335423444
  (0, 13547)	0.02588586723393978
  (0, 24201)	0.027903317660816346
  (0, 20104)	0.027903317660816346
  (0, 32815)	0.021961406106990624
  (0, 63699)	0.027903317660816346
  (0, 23232)	0.030746754335423444
  (0, 4905)	0.030746754335423444
  :	:
  (57, 1305)	0.02678272472